In [1]:
import pandas as pd
import numpy as np
import sklearn as skl
from lightgbm import LGBMRegressor
from challenge_spotter import config as cfg
from challenge_spotter.auxiliars import get_lgbm_baseline,get_random_forest_baseline,get_xgb_baseline,get_pipeline,get_features_dict,run_temporal_cv,prepare_data,get_linear_regresion_baseline,get_pipeline_target_log_transform,show_results
import optuna
from sklearn.linear_model import Ridge


train_df= pd.read_parquet(cfg.DATA_DIR / "train.parquet")
test_df= pd.read_parquet(cfg.DATA_DIR / "test.parquet")

train_df= train_df.sort_values(by="date",ascending=True).reset_index(drop=True)
features_dict= get_features_dict()


c:\Users\kuroc\OneDrive\Escritorio\challenge_spotter\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#The best baseline
X_train , Y_train = prepare_data(train_df)
linear_regresor= get_linear_regresion_baseline()
pipeline= get_pipeline(linear_regresor,features_dict,scale_features=True)
run_temporal_cv(pipeline,X_train,Y_train,show_logs=True,eval_metric="MAE")

results per fold:
  Fold 1: 141.1499
  Fold 2: 128.0751
  Fold 3: 128.6422
  Fold 4: 138.8497
  Fold 5: 134.5878
------
  MAE Mean: 134.2609
  Std: 5.2623
  Min / Max: [128.0751, 141.1499]


array([141.14992777, 128.07505886, 128.64219982, 138.84969618,
       134.58779829])

In [ ]:
#the performance drop applying the log transform to the target ussing the linear model.
X_train , Y_train = prepare_data(train_df)
linear_regresor_log= get_linear_regresion_baseline()
pipeline_linear_regresor_log= get_pipeline(linear_regresor_log,features_dict,scale_features=False)
pipeline_wraper_log_transform= get_pipeline_target_log_transform(pipeline_linear_regresor_log)
run_temporal_cv(pipeline_wraper_log_transform,X_train,Y_train)


In [ ]:
#the random forest had the worst performance in the baseline testing, but improve a lot ussing log transform and some features.
#still being worst than the linear model
X_train , Y_train = prepare_data(train_df)
random_forest= get_random_forest_baseline()
rf_pipeline= get_pipeline(random_forest,features_dict)
rf_pipeline_log_transform= get_pipeline_target_log_transform(rf_pipeline)
run_temporal_cv(rf_pipeline_log_transform,X_train,Y_train)

In [ ]:
#consistenly better than RF, worst than the linear regressor. Also improve a little bit when log transform is applied to the target.
X_train , Y_train = prepare_data(train_df)
xgb_model= get_xgb_baseline()
pipeline_xgb= get_pipeline(xgb_model,features_dict)
pipeline_xgb_log_transform= get_pipeline_target_log_transform(pipeline_xgb)
run_temporal_cv(pipeline_xgb,X_train,Y_train)

In [ ]:
#the second best one. Matching the perfromance of the linear model.
X_train , Y_train = prepare_data(train_df)
lgbm_model= get_lgbm_baseline()
lgbm_pipeline= get_pipeline(lgbm_model,features_dict)
lgbm_log_transform= get_pipeline_target_log_transform(lgbm_pipeline)
run_temporal_cv(lgbm_log_transform,X_train,Y_train)

With that explored, i proceed to search optimal hyperparams for the 2 best models. (Ridge and LGBM).

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_objective_ridge(X_train, Y_train, features_dict):
    def objective_ridge(trial):

        alpha = trial.suggest_float("alpha", 1e-3, 1e4, log=True)

        clusters = trial.suggest_int("clusters", 1, 20, log=True)

        
        linear_model = Ridge(alpha=alpha, random_state=42)

        pipeline = get_pipeline(linear_model,features_dict,scale_features=True,n_clusters=clusters)

        score= run_temporal_cv(pipeline,X_train,Y_train,show_logs=False,eval_metric="MAE")
        
        return score.mean()
    return objective_ridge

X_train , Y_train= prepare_data(train_df)
objective_fn = create_objective_ridge(X_train, Y_train, features_dict)
study_ridge = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=42))
study_ridge.optimize(objective_fn, n_trials=200)

print(f"best score: {study_ridge.best_value:.4f}")
for param, value in study_ridge.best_params.items():
    print(f"  - {param}: {value}")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_objective_ridge(X_train, Y_train, features_dict):
    def objective_ridge(trial):

        n_clusters = trial.suggest_int("n_clusters", 2, 20)
        learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
        num_leaves = trial.suggest_int("num_leaves", 15, 255)
        min_child_samples = trial.suggest_int("min_child_samples", 10, 100)
        n_estimators = trial.suggest_int("n_estimators", 100, 1000)
        subsample = trial.suggest_float("subsample", 0.6, 1.0)
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)

        lgbm_model = LGBMRegressor(
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=-1,
        min_child_samples=min_child_samples,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        )

        lgbm_op_pipeline = get_pipeline(lgbm_model,features_dict,scale_features=True,n_clusters=n_clusters)
        lgbm_op_pipeline_log= get_pipeline_target_log_transform(lgbm_op_pipeline)
        score= run_temporal_cv(lgbm_op_pipeline_log,X_train,Y_train,show_logs=False,eval_metric="MAE")
        return score.mean()
    return objective_ridge

X_train , Y_train= prepare_data(train_df)
objective_fn = create_objective_ridge(X_train, Y_train, features_dict)
study_ridge = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler(seed=42))
study_ridge.optimize(objective_fn, n_trials=200)

print(f"best results :{study_ridge.best_value:.4f}")
for param, value in study_ridge.best_params.items():
    print(f"  - {param}: {value}")

best RMSE :583.5129
  - n_clusters: 3
  - learning_rate: 0.04085923134532415
  - num_leaves: 64
  - max_depth: 2
  - min_child_samples: 56
  - n_estimators: 711
  - subsample: 0.9580712658563583
  - colsample_bytree: 0.8089081041365123

Finally i will run CV with the optimal hyperparams founded and check against test. 

In [4]:
linear_optuna_model = Ridge(alpha=17.269978740046007, random_state=42)
X_train , Y_train = prepare_data(train_df)
ridge_optuna_pipeline= get_pipeline(linear_optuna_model,features_dict,n_clusters=5,scale_features=True)
run_temporal_cv(ridge_optuna_pipeline,X_train,Y_train,eval_metric="MAE")

X_test, Y_test= prepare_data(test_df)

ridge_optuna_pipeline.fit(X_train,Y_train)
ridge_predictions_oos= ridge_optuna_pipeline.predict(X_test)
show_results(y_test=Y_test, y_predicted= ridge_predictions_oos)

results per fold:
  Fold 1: 134.8286
  Fold 2: 126.9011
  Fold 3: 127.0751
  Fold 4: 138.3923
  Fold 5: 133.6565
------
  MAE Mean: 132.1707
  Std: 4.5103
  Min / Max: [126.9011, 138.3923]
----------------
  Test MAE:  148.0048
  Test RMSE: 648.7233


In [ ]:
pd.Series(ridge_predictions_oos).max()

In [ ]:
audit_df = X_test.copy()
audit_df["real_cost"] = Y_test
audit_df["predicted_cost"] = ridge_predictions_oos
audit_df["error"] = np.abs(audit_df["real_cost"] - audit_df["predicted_cost"])

worst_predictions = audit_df.sort_values(by="error", ascending=False)
cols_to_inspect = [
    "pickup",
    "delivery",
    "equipment",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "real_cost",
    "predicted_cost",
    "error",
]

print(worst_predictions[cols_to_inspect].head(15).to_string())
print(f"\nMAE OOS: {audit_df['error'].mean():.2f}")
print(f"Max Error OOS: {audit_df['error'].max():.2f}")

top_20 = worst_predictions[cols_to_inspect].head(30)
p99 = audit_df["real_cost"].quantile(0.99)
top_error_no_outliers= (top_20["real_cost"] < 7000).sum()
print(f"ammong the top 30 worst predictions, the ammount of cases that are not outliers is {top_error_no_outliers}")


In [ ]:
lgbm_model_tuned = LGBMRegressor(
    learning_rate=0.013509291448953806,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=5,
    n_estimators=847,
    subsample=0.8689120872516707,
    colsample_bytree=0.7694118918514066,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,)

X_train , Y_train = prepare_data(train_df)
lgbm_optuna_pipeline= get_pipeline(lgbm_model_tuned,features_dict,n_clusters=5)
log_pipeline= get_pipeline_target_log_transform(lgbm_optuna_pipeline)

run_temporal_cv(lgbm_optuna_pipeline,X_train,Y_train,eval_metric="MAE")



X_test, Y_test= prepare_data(test_df)

lgbm_optuna_pipeline.fit(X=X_train,y=Y_train)
predicts_lgbm_oos=lgbm_optuna_pipeline.predict(X_test)
show_results(y_test=Y_test, y_predicted= predicts_lgbm_oos)


conclusions: The linear model consistenly matched the performance of more complex, tuned models, making it the best option in terms of complexity / performance.
Additionally  the metrics (MAE and RMSE) have a huge gap this is completly caused because RMSE are sensitive to outliers. As shown previously The top 30 worst predictions are all above the percentile 99% in the target scale. And having 130 values over 7000, seems extremely hard to improve the performance in that segment without degrading  the performance in the other 99% of the dataset.

In [3]:
train_df.head(1).to_json()

'{"load_id":{"0":"TR-000001"},"pickup":{"0":"Richmond"},"delivery":{"0":"Baltimore"},"pickup_lat":{"0":38.09122},"pickup_lon":{"0":-76.78906},"delivery_lat":{"0":38.16908},"delivery_lon":{"0":-72.74564},"distance":{"0":274.3},"equipment":{"0":"Dry Van"},"weight":{"0":30658.0},"date":{"0":"2025-01-01"},"market_index":{"0":0.95684},"quote_signal":{"0":2.39595},"posted_rate":{"0":645.41}}'